# 네이버페이 증권 종목토론실 데이터 크롤링
---
파이썬(Python)을 이용한 데이터 분석을 할 때, 흔히들 관심을 가지는 주제로 주가 및 관련 정보 분석이 있습니다. 아무래도 자동 투자 알고리즘 개발 등이 인기있다보니 이를 위한 데이터 수집 및 분석도 관심을 받고 있는 모양입니다.  
</p></br></br>

그래서 많은 사람들이 이용하고 있는 네이버페이 증권의 종목토론실 데이터를 크롤링하고, 이 정보를 감정분석까지 해 보려 합니다. 이번에는 데이터 크롤링 작업만 해 보도록 하며, 해당 페이지는 테이블 태그로 구성되어 있기 때문에 크롤링 난이도가 낮은 편이예요.  
</p></br></br>

## 패키지 안내
---
해당 작업에 이용되는 파이썬 패키지는 requests, BeautifulSoup, pandas 정도가 있습니다. 만약 설치해 두지 않은 패키지가 있다면 사전에 설치해 주시기 바랍니다.  
</p></br></br>

## 종목토론실 URL 분석
---
우선은, 삼성전자 종목토론실의 최신 데이터를 표 형태로 크롤링할 수 있는 코드를 작성해 보겠습니다. 해당 정보는 바로 접속할 수 있는 URL이 있는데요, `https://finance.naver.com/item/board.naver?code=005930&page=1` 형태의 URL로 나타납니다. 여기서, 매개변수에 따라 조회할 수 있는 데이터가 달라지니, 아래 항목을 참조해서 보고 싶은 데이터로 고쳐 적어 주시면 되겠습니다.  
</p></br></br>

* code: 조회할 종목 번호입니다. 만약 삼성전자(005930)라면 code=005930 이라고 적어넣을 수 있겠지요.
* page: 조회할 페이지 번호입니다. 10페이지를 조회하고 싶다면 page=10 이라고 적어넣을 수 있습니다.  
</p></br></br>

## 데이터 크롤링
---
우선은 삼성전자 종목토론실 1쪽의 데이터를 판다스 데이터프레임 형태로 받아오는 코드를 작성해 보도록 하겠습니다. 판다스에서는 html 문서 중에서 table 태그를 모두 파싱한 뒤, 리스트로 반환해주는 `pandas.read_table()` 이라는 함수가 있습니다. 그래서, `requests.get()` 함수를 이용해 네이버페이 종목토론실 html 문서를 받아온 뒤, table 태그를 파싱하도록 합니다. 첫 번째 표는 시가 정보이기 때문에, 두 번째 표를 입력하면 됩니다.  
</p></br></br>


In [1]:
# Import pacakge
import requests
from bs4 import BeautifulSoup
import pandas as pd


CODE = '005930'
PAGE = 1
URL = f"https://finance.naver.com/item/board.naver?code={CODE}&page={PAGE}"

# 접속 정보를 입력
HEADERS = {
    'Referer': 'https://finance.naver.com/item/board.naver?code=005930',
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

response = requests.get(URL, headers=HEADERS)
soup = BeautifulSoup(response.text, 'html.parser')

# 0: 시가 데이터, 1: 종목토론실 데이터
stock_table = pd.read_html(response.text)[1]
stock_table.head()

,날짜,제목,글쓴이,조회,공감,비공감,Unnamed: 6
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,클린봇으로 착한게시글만 모아보세요! 설정,클린봇으로 착한게시글만 모아보세요! 설정,클린봇으로 착한게시글만 모아보세요! 설정,클린봇으로 착한게시글만 모아보세요! 설정,클린봇으로 착한게시글만 모아보세요! 설정,클린봇으로 착한게시글만 모아보세요! 설정,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025.02.24 12:57,삼토방에서 2찍놀이한 연넘들(욕+폭도몰이...,az******,1,0,0,NaN
4,2025.02.24 12:57,여러 대학에서 시국 선언..,hang****,2,0,0,NaN


  
</p></br></br>

여기서, 결측치(NaN) 및 필요 없는 상위 세 줄을 삭제할 수 있도록 코드를 작성해 봅시다.  
</p></br></br>


In [2]:
stock_table = stock_table.iloc[3:,:-1].dropna()
stock_table.head()

,날짜,제목,글쓴이,조회,공감,비공감
3,2025.02.24 12:57,삼토방에서 2찍놀이한 연넘들(욕+폭도몰이...,az******,1,0,0
4,2025.02.24 12:57,여러 대학에서 시국 선언..,hang****,2,0,0
5,2025.02.24 12:57,클린봇이 이용자 보호를 위해 숨긴 게시글입니다.,blan****,1,0,1
6,2025.02.24 12:57,찢투세 때문에 시장 다죽여 놓은거 잊지말...,ji******,4,0,1
7,2025.02.24 12:56,"민주노총에는 간첩이 우굴거린다,, 며칠...",anje****,5,1,2


  
</p></br></br>

## 여러 페이지 크롤링하기
---
만약 여러 페이지 정보를 함께 크롤링하고 싶다면, 반복문을 이용해서 위 작업을 계속할 수 있습니다. 이번에는 간단하게, 1~10쪽의 데이터를 하나의 표로 정리하는 코드를 작성해 보도록 하겠습니다.  
</p></br></br>


In [3]:
stock_table_10 = pd.DataFrame()

for PAGE in range(1,11):
    URL = f"https://finance.naver.com/item/board.naver?code={CODE}&page={PAGE}"
    
    response = requests.get(URL, headers=HEADERS)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    stock_table_temp = pd.read_html(response.text)[1].iloc[3:,:-1].dropna()
    stock_table_10 = pd.concat([stock_table_10, stock_table_temp])

stock_table_10.reset_index(drop=True, inplace=True)  # 인덱스 초기화
stock_table_10.tail()

,날짜,제목,글쓴이,조회,공감,비공감
195,2025.02.24 11:07,캐에잡주 던져라!,dmka****,13,0,1
196,2025.02.24 11:05,"탄핵의 진실, 체제전쟁",bigb****,27,3,2
197,2025.02.24 11:04,여러분 꿈깨라고한건 [7],cmj1****,76,0,0
198,2025.02.24 11:03,클린봇이 이용자 보호를 위해 숨긴 게시글입니다.,yaim****,7,2,0
199,2025.02.24 11:03,요한이 가로되ㅡ 보라 ㅣ ㅎ ㅏ나님의 어...,true****,18,0,1


  
</p></br></br>

이제, 해당 데이터를 이용하거나 추가로 다른 데이터를 병용해서 주가 예측 또는 주식 거래량 예측 등의 작업을 해볼 수 있습니다.  
  
</p></br></br>
